# 10. Predict, check and explain

**Did the tree improve predictions for the same farms?** Use the final tree from Chapter 9 on the 21 held-out farms. Start with the same milk sample you checked by hand in Chapter 7, then compare both methods on all 105 records.

**Open.** Restart R and save the supplied script as `scripts/my-10-test-tree.R`. This chapter uses `outputs/checkpoints/tree-fit.rds` from Chapter 9. Run the named blocks in order.

## 10.1 Load and read the final tree

Run {span .script-block}`BLOCK · load-rpart` and {span .script-block}`BLOCK · load-tree-fit` to restore the final tree and its training medians. If the checkpoint is missing, complete the save step in Chapter 9.

In [1]:
library(rpart)

{span .script-block}`BLOCK · load-tree-fit`

In [2]:
source("scripts/tree_helpers.R")
if (!file.exists("outputs/checkpoints/tree-fit.rds")) stop("Run Chapter 9 first to create tree-fit.rds.")
chosen <- readRDS("outputs/checkpoints/tree-fit.rds")
final_tree <- chosen$fit
train_filled <- fill_predictors(chosen$train, chosen$medians)

Run {span .script-block}`BLOCK · plot-tree`. The diagram is also saved at `outputs/tree/fitted-tree.png`.

In [3]:
nodes <- extract_tree_nodes(final_tree)
write_tree_mermaid(nodes, "outputs/tree/fitted-tree.mmd")
png("outputs/tree/fitted-tree.png", width = 1200, height = 720, res = 150)
par(mar = c(0, 0, 3, 0), xpd = NA, family = "sans")
draw_teaching_tree(final_tree, nodes)
invisible(dev.off())
cat("Saved outputs/tree/fitted-tree.png\n")

Saved outputs/tree/fitted-tree.png


![The final tree asks whether the farm has at least 36.5 milking cows. The leaves show the mean log10 MSC and number of training records in each group.](../outputs/tree/fitted-tree.png)

Chapter 9 selected a maximum depth of 1. Look at the fitted tree. **Which input determines the branch? Where does the number in each leaf come from?**

```{admonition} Check your reading of the tree
:class: tip dropdown

The tree splits on milking-cow count. The cutoff of 36.5 separates herds of 36 cows or fewer from herds of 37 or more. The prediction in each leaf is the mean log10 MSC of the training records that reached it.
```

## 10.2 Compare two predictions for one real record

**Study data.** Find **farm R15, sampling round A**. This is the first test record you checked by hand with the baseline in [Section 7.2](07-baseline.ipynb#id-7-2-check-one-prediction-by-hand). The farm had **50 milking cows** that day. Follow the tree diagram to find its leaf and prediction before running the blocks.

Run {span .script-block}`BLOCK · prepare-test` to fill missing test inputs with the saved training medians.

In [4]:
# Test inputs use saved training medians, with no refit or new choice.
test_filled <- fill_predictors(chosen$test, chosen$medians)

Run {span .script-block}`BLOCK · predict-tree` to make the test predictions.

In [5]:
tree_prediction <- predict(final_tree, newdata = test_filled)

Run {span .script-block}`BLOCK · trace-one-test-prediction`. Check the farm ID, sampling round and leaf. For R15/A, the recorded cow count and the count used by the model are both **50**. The tree sends the record to **leaf 2**.

In [6]:
first_tree_test_record <- test_filled[1, ]
first_route <- trace_tree(first_tree_test_record, nodes)
data.frame(feature = tree_features, recorded = unlist(chosen$test[1, tree_features], use.names = FALSE),
           used_by_model = unlist(first_tree_test_record[tree_features], use.names = FALSE))
data.frame(FarmID = first_tree_test_record$FarmID, Sampling = first_tree_test_record$Sampling,
           observed_log10 = first_tree_test_record$log10_msc,
           rule = paste(first_route$path, collapse = "; "), leaf = first_route$leaf,
           prediction_log10 = first_route$prediction)
stopifnot(isTRUE(all.equal(first_route$prediction, as.numeric(tree_prediction[1]))))

feature,recorded,used_by_model
<chr>,<dbl>,<dbl>
pasture_hours,12,12
milking_cows,50,50


FarmID,Sampling,observed_log10,rule,leaf,prediction_log10
<chr>,<chr>,<dbl>,<chr>,<int>,<dbl>
R15,A,0.8129134,milking_cows >= 36.5,2,0.6873386


**Work it out.** The lab result is about **0.8129** on the log10 scale. The baseline predicts **0.7450**, and the tree predicts **0.6873**. For each method, subtract the prediction from the same lab result. Which prediction is closer?

Run {span .script-block}`BLOCK · check-one-test-residual` after you try. Its last column shows both residuals.

In [7]:
baseline_prediction <- rep(mean(chosen$train$log10_msc), nrow(test_filled))
first_observed <- first_tree_test_record$log10_msc
first_predictions <- c(baseline_prediction[1], as.numeric(tree_prediction[1]))
first_test_comparison <- data.frame(
  model = c("Training-mean baseline", "Shallow regression tree"),
  observed_log10 = first_observed,
  predicted_log10 = first_predictions,
  residual_log10 = first_observed - first_predictions)
first_test_comparison

model,observed_log10,predicted_log10,residual_log10
<chr>,<dbl>,<dbl>,<dbl>
Training-mean baseline,0.8129134,0.7450425,0.06787081
Shallow regression tree,0.8129134,0.6873386,0.12557474


The baseline missed by about **+0.0679** and the tree by about **+0.1256**. Both predictions were lower than the measured value. The baseline was closer on this record.

Run {span .script-block}`BLOCK · check-leaves` to see where the tree’s prediction comes from. The **354 training records** that reached leaf 2 have a mean of about **0.6873** on the log10 scale.

In [8]:
train_routes <- lapply(seq_len(nrow(train_filled)), function(i) trace_tree(train_filled[i, ], nodes))
train_leaf <- vapply(train_routes, function(route) route$leaf, integer(1))
train_prediction <- as.numeric(predict(final_tree, newdata = train_filled))
stopifnot(isTRUE(all.equal(vapply(train_routes, function(route) route$prediction, numeric(1)), train_prediction)))
# Group the training outcomes by leaf and calculate one mean per leaf.
leaf_means <- aggregate(train_filled$log10_msc, list(leaf = train_leaf), mean)
names(leaf_means)[2] <- "mean_from_training_records"
leaf_means$tree_prediction <- nodes$mean_log10[match(leaf_means$leaf, nodes$node)]
stopifnot(isTRUE(all.equal(leaf_means$mean_from_training_records, leaf_means$tree_prediction)))
leaf_means

leaf,mean_from_training_records,tree_prediction
<int>,<dbl>,<dbl>
2,0.6873386,0.6873386
3,1.0545454,1.0545454


## 10.3 Compare all 105 predictions and write a conclusion

**Does the baseline also come closer across the whole test set?** Run {span .script-block}`BLOCK · tree-errors` to repeat your subtraction for every test record.

In [9]:
baseline_errors <- test_filled$log10_msc - baseline_prediction
tree_errors <- test_filled$log10_msc - tree_prediction

Run {span .script-block}`BLOCK · tree-test-scores`. Compare the two rows. Both methods use the same **105 records from 21 test farms**, and both scores use the log10 CFU/mL scale.

In [10]:
tree_scores <- data.frame(model = c("Training-mean baseline", "Shallow regression tree"),
  maxdepth = c(0, chosen$chosen_depth), n_test = nrow(test_filled), test_farms = length(unique(test_filled$FarmID)),
  RMSE_log10_CFU_ml = c(sqrt(mean(baseline_errors^2)), sqrt(mean(tree_errors^2))),
  MAE_log10_CFU_ml = c(mean(abs(baseline_errors)), mean(abs(tree_errors))))
tree_scores[c("model", "n_test", "test_farms", "RMSE_log10_CFU_ml", "MAE_log10_CFU_ml")]

model,n_test,test_farms,RMSE_log10_CFU_ml,MAE_log10_CFU_ml
<chr>,<int>,<int>,<dbl>,<dbl>
Training-mean baseline,105,21,0.5289231,0.4091480
Shallow regression tree,105,21,0.5448970,0.4264264


The baseline has the lower **RMSE** (about **0.529** versus **0.545**) and **MAE** (about **0.409** versus **0.426**). On these held-out records, the training-mean prediction performed better than the tree.

Run {span .script-block}`BLOCK · save-tree-results`. In **Files → outputs → tree**, open `test-scores.csv` and `test-predictions.csv`. The first file gives the comparison. The second lets you find R15/A and check both predictions against its lab result.

In [11]:
write.csv(tree_scores, "outputs/tree/test-scores.csv", row.names = FALSE)
write.csv(nodes, "outputs/tree/nodes.csv", row.names = FALSE)
write.csv(leaf_means, "outputs/tree/leaf-means.csv", row.names = FALSE)
write.csv(data.frame(feature = tree_features, median = unname(chosen$medians)),
          "outputs/tree/training-medians.csv", row.names = FALSE)
training_predictions <- data.frame(FarmID = train_filled$FarmID, Sampling = train_filled$Sampling,
  observed_log10 = train_filled$log10_msc, train_filled[tree_features], leaf = train_leaf,
  tree_prediction = train_prediction)
write.csv(training_predictions, "outputs/tree/training-predictions.csv", row.names = FALSE)
test_predictions <- data.frame(FarmID = test_filled$FarmID, Sampling = test_filled$Sampling,
  observed_log10 = test_filled$log10_msc, test_filled[tree_features],
  pasture_hours_missing = is.na(chosen$test$pasture_hours), milking_cows_missing = is.na(chosen$test$milking_cows),
  tree_prediction = as.numeric(tree_prediction), baseline_prediction = baseline_prediction)
write.csv(test_predictions, "outputs/tree/test-predictions.csv", row.names = FALSE)
writeLines(capture.output(sessionInfo()), "outputs/tree/session-info.txt")

**Write three sentences in English.** Name the candidate inputs and the depth chosen in Chapter 9. Compare the two methods on the test records. Then explain one limitation that matters to the result.

```{admonition} Reference conclusion — open after you try
:class: tip dropdown

We offered pasture hours and milking-cow count as inputs, and five-fold validation within the training farms selected depth 1. On 105 records from 21 held-out farms, the baseline had the lower RMSE, about 0.529 versus 0.545 for the tree on the log10 CFU/mL scale. The tree’s score includes predictions for records whose missing survey inputs were filled with training medians.
```

You have completed the test comparison when you can show the score table and explain your conclusion. Next, save the tree for another prediction and plan how to repeat the workflow with a different model.

## 10.4 Save the model and try a new input

Chapter 9 saved `tree-fit.rds` to carry the fitted tree and the study tables into this chapter. For future predictions, `model.rds` bundles the **fitted tree, training medians and input names**. The medians are part of the prediction process because a new profile can have missing inputs.

Run {span .script-block}`BLOCK · save-model` to write `outputs/tree/model.rds`. Then run {span .script-block}`BLOCK · reload-model` to read it back as `saved_model`.

In [12]:
model_for_reuse <- list(fit = final_tree, medians = chosen$medians, features = tree_features)
saveRDS(model_for_reuse, "outputs/tree/model.rds")

{span .script-block}`BLOCK · reload-model`

In [13]:
saved_model <- readRDS("outputs/tree/model.rds")

```{admonition} Invented data — try the saved tree on farm profiles
:class: note

The two practice profiles have six hours of pasture time and cow counts of 50 and 30. They have no measured MSC. Use them to see how the saved rule makes predictions.
```

Run {span .script-block}`BLOCK · new-samples`, {span .script-block}`BLOCK · prepare-new` and {span .script-block}`BLOCK · new-predict`. The preparation uses `saved_model$medians`. Prediction uses `saved_model$fit`.

In [14]:
# Invented input profiles, without observed outcomes; not an evaluation dataset.
new_samples <- data.frame(pasture_hours = c(6, 6), milking_cows = c(50, 30))
new_samples

pasture_hours,milking_cows
<dbl>,<dbl>
6,50
6,30


{span .script-block}`BLOCK · prepare-new`

In [15]:
new_ready <- fill_predictors(new_samples, saved_model$medians)

{span .script-block}`BLOCK · new-predict`

In [16]:
new_prediction <- predict(saved_model$fit, newdata = new_ready)
example_predictions <- data.frame(new_samples, prediction_log10 = as.numeric(new_prediction),
                                  concentration_expression_CFU_ml = 10^as.numeric(new_prediction))
example_predictions
write.csv(example_predictions, "outputs/tree/example-predictions.csv", row.names = FALSE)

pasture_hours,milking_cows,prediction_log10,concentration_expression_CFU_ml
<dbl>,<dbl>,<dbl>,<dbl>
6,50,0.6873386,4.867866
6,30,1.0545454,11.338235


**Change one input.** In your my- copy, change **30 cows to 37** in {span .script-block}`BLOCK · new-samples`. Which side of the cutoff will that profile reach? Rerun **new-samples → prepare-new → new-predict** and check your guess. The last column gives the same prediction in CFU/mL using `10^prediction`.

```{admonition} Check the changed profile
:class: tip dropdown

At 30 cows, the profile reaches leaf 3 and receives about 1.0545 on the log10 scale. At 37 cows it reaches leaf 2, so the prediction becomes about 0.6873.
```

## 10.5 Take this workflow to the next model

Imagine you want to try another model on the milk study. Fill in these four lines before asking AI for code. Use your tree workflow as the example.

| Decision | What to write down |
|---|---|
| **What will I predict?** | What does one row represent? Which column is the answer? Which inputs will be available when I predict? |
| **How will I split the data?** | Which records stay together? Which farms choose settings, and which farms are reserved for the final comparison? |
| **How will the model learn?** | What preparation and fitting steps will I learn from training records? How will I choose any model settings? |
| **How will I compare results?** | Which predictions and lab results will I compare? What will I calculate and save? |

```{admonition} Reference worksheet for the tree
:class: tip dropdown

1. Predict log10 MSC for one farm and sampling round using the two survey inputs available before the lab result, as assumed in this exercise.
2. Keep each farm’s records together. Choose settings within the 81 training farms and reserve 21 farms for testing.
3. Learn missing-input references inside each fitting group. Choose tree depth by validation, then refit on all training farms.
4. Compare the tree and baseline on the same 105 test records using RMSE and MAE. Save the scores and the individual predictions.
```

To replay the full reference analysis, restart R and run a my- copy of `scripts/full-analysis.R`. To check an edit of your own, run the changed blocks in your chapter script.

## 10.6 Optional AI check

Return to the **study data** and ask AI to repeat two familiar operations: fit the tree using the settings chosen in Chapter 9, then predict the same held-out records. Use your my- copy and a browser chat from [AI access](01-first-run.ipynb#id-1-5-ai-access).

Run {span .script-block}`BLOCK · restore-selected-tree-settings` so the chosen settings are available as `selected_settings`.

In [17]:
selected_settings <- fixed_tree_settings
selected_settings$maxdepth <- chosen$chosen_depth
# My live AI attempt: add returned code and your check below in your my- copy.

```text
I am using R and rpart. Fit a numeric regression tree called ai_tree
using train_filled, with log10_msc as the outcome and pasture_hours
and milking_cows as the two inputs. Use method = "anova" and
control = selected_settings. Then use predict() on test_filled and
store the results as ai_prediction. Use the existing split, settings
and prepared tables. Explain which line fits and which predicts.
```

Read the reply before running it. The fitting call should use `train_filled` and `selected_settings`. The prediction call should use `test_filled`. In your my- script, add the returned R code below **My live AI attempt**, then run it and check:

```r
all.equal(as.numeric(ai_prediction), as.numeric(tree_prediction))
```

Matching predictions give **TRUE**. If you see a difference or error, compare the code and output with the supplied blocks. Save your prompt, reply and check in your AI-use log.

**Code-inspection route.** For the same exercise without live chat, find the fitting call in {span .script-block}`BLOCK · fit-tree` in Chapter 9 and the prediction call in {span .script-block}`BLOCK · predict-tree` here. Use the checklist above to explain the role of each table and setting. Record the route as **code inspection**.